# 01 - Simulation Basics And Metric Contract

Use this notebook first. It shows the shortest path through the codebase:
`YAML config -> SimulationConfig -> ClinicAppointmentSimulation -> SimulationResults -> shared analysis metrics`.

The goal is to understand what the engine returns and which metric helpers should be used by reports and experiments.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_DIR = Path.cwd()
if not (REPO_DIR / "engine_files").exists():
    REPO_DIR = REPO_DIR.parent
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import (
    METRIC_DEFINITIONS,
    aggregate_result_row,
    class_result_rows,
    metric_definition_rows,
    outcome_rates_from_result,
    outcome_totals,
    result_metrics_from_result,
)
from analysis.plot_style import (
    BASELINE_COLOR,
    driver_heatmap_cmap,
    driver_line_style,
    plot_driver_line,
)
from engine_files.config_loader import load_config
from engine_files.engine import ClinicAppointmentSimulation
from engine_files.model import ThresholdRule

BASE_CONFIG = load_config(REPO_DIR / "configs" / "baseline.yaml")
pd.options.display.max_columns = 120

## Inspect The Baseline Config

The config object is immutable enough for safe experimentation with `dataclasses.replace`.
Each class owns arrival rate, balking rule, cancellation probability, no-show rule, and value.

In [ ]:
config = BASE_CONFIG

config_summary = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "lambda_per_day": params.lambda_per_day,
            "balk_threshold": params.balk_prob.threshold,
            "balk_low": params.balk_prob.low,
            "balk_high": params.balk_prob.high,
            "cancel_prob": params.cancel_prob,
            "no_show_threshold": params.no_show_prob.threshold,
            "no_show_low": params.no_show_prob.low,
            "no_show_high": params.no_show_prob.high,
            "value": params.value,
        }
        for class_id, params in config.classes.items()
    ]
)

display(
    pd.Series(
        {
            "slots_per_day": config.slots_per_day,
            "horizon_days": config.horizon_days,
            "burn_in_days": config.burn_in_days,
            "measure_days": config.measure_days,
            "cooldown_days": config.cooldown_days,
            "seed": config.seed,
        },
        name="baseline",
    ).to_frame()
)
display(config_summary)

## Run One Simulation

`result_metrics_from_result` is the canonical two-class metric contract for the study.
`class_result_rows` and `aggregate_result_row` are the row builders used by sweeps.

In [ ]:
result = ClinicAppointmentSimulation(config).run()

metric_table = pd.Series(result_metrics_from_result(result), name="value").to_frame()
class_table = pd.DataFrame(class_result_rows(result))
aggregate_row = pd.Series(aggregate_result_row(result), name="value").to_frame()

display(metric_table)
display(class_table)
display(aggregate_row)

## Metric Definitions

These names are the ones the report and notebooks should use consistently.

In [ ]:
pd.DataFrame(metric_definition_rows())

## Outcome Rates And Measurement Semantics

`unresolved_booked` is a diagnostic for late measurement-window bookings that have not resolved by simulation end.
The current model documents this caveat but does not change engine behavior.

In [ ]:
totals = outcome_totals(result)
rates = outcome_rates_from_result(result)

display(pd.Series(totals, name="count").to_frame())
display(pd.Series(rates, name="rate").to_frame())
print(f"cooldown_days = {config.cooldown_days}; horizon_days - 1 = {config.horizon_days - 1}")

## Inspect Calendar State

The final calendar stores one row per residual day and one entry per daily slot.
A tuple `(class_id, tau)` means a booked patient and their original offered delay.

In [ ]:
final_calendar = pd.DataFrame(result.final_full_state)
print(f"final calendar shape: {final_calendar.shape}")
display(final_calendar.head())

print("first two start-of-day summary states:")
for state in result.daily_summary_states[:2]:
    display(pd.DataFrame(state))

## Small Seed Check

Use this cell to see simulation noise before running larger sweeps.

In [ ]:
seed_rows = []
for seed in [11, 12, 13, 14, 15]:
    seeded_config = replace(config, seed=seed)
    seeded_result = ClinicAppointmentSimulation(seeded_config).run()
    seed_rows.append({"seed": seed, **result_metrics_from_result(seeded_result)})

seed_df = pd.DataFrame(seed_rows)
display(seed_df[["seed", "average_utilization", "overall_percent_serviced", "mean_offered_booking_delay", "overall_balking_rate", "access_advantage_class_1"]])
seed_df.describe(numeric_only=True).T